# Install dependencies

In [ ]:
!pip install natsort
!git clone https://github.com/JaidedAI/EasyOCR.git
# set particular version of easyocr repo (1.7.0)
!git checkout f947eaa36a55adb306feac58966378e01cc67f85
%cd EasyOCR/trainer

# for cer metric
!pip install jiwer

# for download easyocr pretrained model from google drive
!pip install gdown

# Imports

In [ ]:
import os
from os.path import join as opj

import cv2
import yaml
import gdown
import torch
import shutil
import easyocr
import numpy as np
import pandas as pd
from PIL import Image
from lxml import etree
from copy import deepcopy
from tqdm.auto import tqdm
from datasets import load_metric
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split

torch.backends.cudnn.benchmark = True # enable, because model will not change and it`s input sizes remain the same

In [ ]:
# disable wandb
import wandb
wandb.init(mode="disabled")

In [ ]:
# rewriting source easyocr scripts to disable multiprocessing and fix bugs of new pytorch version

content = open("train.py").read()
content = content.replace("from test import validation", 
"""from importlib.machinery import SourceFileLoader
test = SourceFileLoader("test","test.py").load_module()
validation = test.validation""")
content = content.replace("prefetch_factor=512", "prefetch_factor=None")
with open("train.py", "w") as f:
    f.write(content)
    
content = open("dataset.py").read()
content = content.replace("data_loader_iter.next()", "next(data_loader_iter)")
content = content.replace("self.dataloader_iter_list[i].next()", "next(self.dataloader_iter_list[i])")
with open("dataset.py", "w") as f:
    f.write(content)

In [ ]:
# import source easyocr code to train model
from train import train as ocr_train
from utils import AttrDict

# Constants

In [ ]:
DATASET_DIR = "/kaggle/input/1200-images-and-ocr-annotations-rus"
VAL_PERCENT, TEST_PERCENT = 0.05, 0.025 # dataset is divided into train, validation and test according to these values multiplied by 100%. TRAIN_PERCENT = 1 - (VAL_PERCENT + TEST_PERCENT)
OUT_MODEL_PATH = "/kaggle/working/model" # path to save trained model
os.makedirs(OUT_MODEL_PATH, exist_ok=True)
PREDICTION_ALLOW_LIST = "qwertyuiopasdfghjklzxcvbnmйцукенгшщзхъфывапролджэячсмитьбю-*" # allowed characters to predict when testing (paragraph "Test")

# TEXTS_DATASET_DIR = "./all_data" # folder of new dataset for easyocr
TEXTS_DATASET_DIR = "/kaggle/working/easyocr_train" # folder of new dataset for easyocr

EASY_OCR_PRETRAINED_URL = "https://drive.google.com/uc?id=1PIywV9_WZqNNfUIk6-bs598fX7OZTcbY" # url to google drive of pretrained easyocr model

EXPERIMENT_NAME = "ru_ocr" # custom name of model train/run
IMG_SIZE = (600, 64) # width and height of image size to train easyocr
NUM_TRAIN_ITERATIONS = 5000 # number of iterations to train. The number of iterations to use depends on the size of the dataset. Too many iterations can lead to overfitting, which can be detected by monitoring the validation loss during training. Too few iterations can result in underfitting, which can be identified by a consistently "sharp" decrease in the loss.
VAL_INTERVAL = 100 # evaluate model each X iterations
BATCH_SIZE = 64 # batch size per iteration. It is ideally in the form of a power of two (2, 4, 8)

# Make new dataset with cropped text images

In [ ]:
import os
import pandas as pd
import shutil
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Пути
IMAGES_DIR = "/kaggle/input/datasets/kapitanboli/easyocr-v4/v4/images"  # Папка с изображениями
LABELS_FILE = "/kaggle/input/datasets/kapitanboli/easyocr-v4/v4/labels.csv"  # Ваш labels.csv файл
OUT_DIR = "/kaggle/working/easyocr_train"  # Выходная директория

# Параметры разделения
VAL_PERCENT = 0.15
TEST_PERCENT = 0.15

os.makedirs(OUT_DIR, exist_ok=True)

# Читаем labels.csv с правильным разделителем
print("Чтение labels.csv...")
# Пробуем разные варианты разделителей
try:
    # Сначала пробуем с точкой с запятой
    df = pd.read_csv(LABELS_FILE, sep=';', header=None, names=['filename', 'words'])
    print("Файл прочитан с разделителем ';'")
except:
    try:
        # Пробуем с табуляцией
        df = pd.read_csv(LABELS_FILE, sep='\t', header=None, names=['filename', 'words'])
        print("Файл прочитан с разделителем '\\t'")
    except:
        # Автоматическое определение
        df = pd.read_csv(LABELS_FILE, sep=None, engine='python', header=None, names=['filename', 'words'])
        print("Разделитель определен автоматически")

print(f"Загружено {len(df)} записей")
print(f"Колонки: {df.columns.tolist()}")
print(f"Примеры:")
print(df.head())

# Очищаем данные
df['filename'] = df['filename'].str.strip()
df['words'] = df['words'].fillna('').str.strip()
df['words'] = df['words'].str.lower()

# Удаляем пустые строки
df = df[df['words'] != '']
print(f"После удаления пустых строк: {len(df)} записей")

# Проверяем существование изображений
print("\nПроверка изображений...")
missing_images = []
for filename in tqdm(df['filename'], desc="Проверка"):
    if not os.path.exists(os.path.join(IMAGES_DIR, filename)):
        missing_images.append(filename)

if missing_images:
    print(f"Внимание: {len(missing_images)} изображений не найдено!")
    if len(missing_images) <= 10:
        print(f"Отсутствующие: {missing_images}")
    else:
        print(f"Примеры отсутствующих: {missing_images[:5]}")
    
    # Удаляем записи с отсутствующими изображениями
    df = df[~df['filename'].isin(missing_images)]
    print(f"Осталось {len(df)} записей после фильтрации")
else:
    print("Все изображения найдены!")

# Разделяем на train/val/test
print("\nРазделение на train/val/test...")
if len(df) < 3:
    print("Ошибка: недостаточно данных для разделения!")
    print("Нужно минимум 3 записи")
else:
    train_df, nontrain_df = train_test_split(
        df, 
        test_size=VAL_PERCENT + TEST_PERCENT, 
        random_state=42
    )
    eval_df, test_df = train_test_split(
        nontrain_df, 
        test_size=TEST_PERCENT / (VAL_PERCENT + TEST_PERCENT), 
        random_state=42
    )

    print(f"Train: {len(train_df)} изображений")
    print(f"Val: {len(eval_df)} изображений")
    print(f"Test: {len(test_df)} изображений")

    # Функция для сохранения подмножества
    def save_subset(subset_df, subset_name, base_dir):
        """Копирует изображения и создает labels.csv для подмножества"""
        subset_dir = os.path.join(base_dir, subset_name)
        os.makedirs(subset_dir, exist_ok=True)
        
        # Копируем изображения
        copied_count = 0
        for _, row in tqdm(subset_df.iterrows(), total=len(subset_df), 
                           desc=f"Копирование {subset_name}"):
            src_path = os.path.join(IMAGES_DIR, row['filename'])
            dst_path = os.path.join(subset_dir, row['filename'])
            
            if os.path.exists(src_path):
                shutil.copy2(src_path, dst_path)
                copied_count += 1
        
        # Создаем labels.csv с правильным форматом
        labels_df = subset_df[['filename', 'words']].copy()
        labels_path = os.path.join(subset_dir, 'labels.csv')
        labels_df.to_csv(labels_path, index=False)
        
        print(f"{subset_name}: скопировано {copied_count} из {len(subset_df)} изображений")
        return copied_count

    # Сохраняем все подмножества
    print("\nСохранение данных...")
    train_copied = save_subset(train_df, "train", OUT_DIR)
    val_copied = save_subset(eval_df, "val", OUT_DIR)
    test_copied = save_subset(test_df, "test", OUT_DIR)

    # Проверяем результат
    print("\n" + "="*60)
    print("РЕЗУЛЬТАТ:")
    print("="*60)

    for subset in ["train", "val", "test"]:
        subset_dir = os.path.join(OUT_DIR, subset)
        labels_file = os.path.join(subset_dir, "labels.csv")
        
        if os.path.exists(labels_file):
            df_check = pd.read_csv(labels_file)
            num_images = len([f for f in os.listdir(subset_dir) 
                             if f.endswith(('.png', '.jpg', '.jpeg'))])
            print(f"\n{subset.upper()}:")
            print(f"  - Изображений в папке: {num_images}")
            print(f"  - Записей в labels.csv: {len(df_check)}")
            if len(df_check) > 0:
                print(f"  - Примеры записей:")
                for i in range(min(3, len(df_check))):
                    print(f"    {df_check.iloc[i]['filename']}: {df_check.iloc[i]['words']}")

    print("\n" + "="*60)
    print("СТРУКТУРА ДАННЫХ:")
    print(f"{OUT_DIR}/")
    print(f"  ├── train/")
    print(f"  │   ├── images...")
    print(f"  │   └── labels.csv")
    print(f"  ├── val/")
    print(f"  │   ├── images...")
    print(f"  │   └── labels.csv")
    print(f"  └── test/")
    print(f"      ├── images...")
    print(f"      └── labels.csv")

    print("\n" + "="*60)
    print("НАСТРОЙКИ ДЛЯ ОБУЧЕНИЯ:")
    print("="*60)
    print(f"""
opt["train_data"] = "{OUT_DIR}"
opt["valid_data"] = "{os.path.join(OUT_DIR, 'val')}"
opt["select_data"] = "train"
    """)

    print("Готово! Можно запускать обучение.")

# Training

## Set model config

In [ ]:
# read easyocr config file
def get_config(file_path):
    with open(file_path, 'r', encoding="utf8") as stream:
        opt = yaml.safe_load(stream)
    opt = AttrDict(opt)
    return opt

In [ ]:
# download pretrained easyocr model
pretrained_easy_ocr_path = 'pretrained_easy_ocr.zip'
gdown.download(EASY_OCR_PRETRAINED_URL, pretrained_easy_ocr_path, quiet=False)

In [ ]:
experiment_path = f'./saved_models/{EXPERIMENT_NAME}'
os.makedirs(experiment_path, exist_ok=True)

In [ ]:
# load basic config
opt = get_config("config_files/en_filtered_config.yaml")

# remake config for finetuning cyrillic model
opt["symbol"] = "!\"#$%&'()*+,-./:;<=>?@[\\]№_`{|}~ €₽"
opt["lang_char"] = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyzАБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯабвгдеёжзийклмнопрстуфхцчшщъыьэюяЂђЃѓЄєІіЇїЈјЉљЊњЋћЌќЎўЏџҐґҒғҚқҮүҲҳҶҷӀӏӢӣӨөӮӯ"
opt["experiment_name"] = EXPERIMENT_NAME
opt["train_data"] = TEXTS_DATASET_DIR
opt["select_data"] = "train"
opt["valid_data"] = opj(TEXTS_DATASET_DIR, "val")
opt["num_iter"] = NUM_TRAIN_ITERATIONS
opt["batch_size"] = BATCH_SIZE
opt["batch_max_length"] = BATCH_SIZE
opt["FT"] = True # enable finetuning
opt["new_prediction"] = False # false to not start training from scratch
opt["saved_model"] = pretrained_easy_ocr_path # path to model for finetuning
opt["valInterval"] = VAL_INTERVAL
opt["batch_ratio"] = "1"
opt["workers"] = 0 # 0 workers to disable multiprocessing
opt["imgW"], opt["imgH"] = IMG_SIZE
opt["character"] = opt["number"] + opt["symbol"] + opt["lang_char"]

## Train

In [ ]:
# train model
try:
    ocr_train(opt, amp=False)
except SystemExit:
    pass

## Save model to exact path

In [ ]:
# save config file
saved_model_opt = {
    "lang_list": ["ru"],
    "network_params": {
        "input_channel": opt["input_channel"],
        "output_channel": opt["output_channel"],
        "hidden_size": opt["hidden_size"]
    },
    "imgH": opt["imgH"],
    "character_list": opt["character"],
}
with open(opj(OUT_MODEL_PATH, EXPERIMENT_NAME + ".yaml"), "w") as outfile:
    yaml.dump(saved_model_opt, outfile, default_flow_style=False)

In [ ]:
# save best weights
shutil.copy(opj(experiment_path, "best_norm_ED.pth"), opj(OUT_MODEL_PATH, EXPERIMENT_NAME + ".pth"))

In [ ]:
# save module of classes to load weights
content = open("model.py").read()
content = content.replace("def __init__(self, opt):",
f"""def __init__(self, *args, **kwargs):
        opt = AttrDict({opt})""")
content = """class AttrDict(dict):
    def __init__(self, *args, **kwargs):
        super(AttrDict, self).__init__(*args, **kwargs)
        self.__dict__ = self
""" + content
with open(opj(OUT_MODEL_PATH, EXPERIMENT_NAME) + ".py", "w") as f:
    f.write(content)

shutil.copytree("modules", opj(OUT_MODEL_PATH, "modules"))

In [2]:
!zip -r file.zip /kaggle/working/model

updating: kaggle/working/model/ (stored 0%)
updating: kaggle/working/model/ru_ocr.pth (deflated 7%)
updating: kaggle/working/model/modules/ (stored 0%)
updating: kaggle/working/model/modules/__pycache__/ (stored 0%)
updating: kaggle/working/model/modules/__pycache__/prediction.cpython-310.pyc (deflated 46%)
updating: kaggle/working/model/modules/__pycache__/transformation.cpython-310.pyc (deflated 51%)
updating: kaggle/working/model/modules/__pycache__/feature_extraction.cpython-310.pyc (deflated 58%)
updating: kaggle/working/model/modules/__pycache__/sequence_modeling.cpython-310.pyc (deflated 40%)
updating: kaggle/working/model/modules/sequence_modeling.py (deflated 60%)
updating: kaggle/working/model/modules/feature_extraction.py (deflated 82%)
updating: kaggle/working/model/modules/prediction.py (deflated 71%)
updating: kaggle/working/model/modules/transformation.py (deflated 73%)
updating: kaggle/working/model/ru_ocr.yaml (deflated 52%)
updating: kaggle/working/model/ru_ocr.py (de

In [3]:
from IPython.display import FileLink
FileLink(r'file.zip')

/kaggle/working/file.zip

# Test

In [ ]:
reader = easyocr.Reader(["ru"], 
                        model_storage_directory=OUT_MODEL_PATH,
                        user_network_directory=OUT_MODEL_PATH,
                        recog_network=EXPERIMENT_NAME)

In [ ]:
instance_path = opj(TEXTS_DATASET_DIR, "test", os.listdir(opj(TEXTS_DATASET_DIR, "test"))[3])
result = reader.readtext(instance_path, allowlist=PREDICTION_ALLOW_LIST)
txt = "\n".join([elem[1] for elem in result])
print(txt)

In [ ]:
Image.open(instance_path)